In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Define data transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),    # Resize images to 224x224
    transforms.ToTensor(),            # Convert images to tensor
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # Normalize images
])

# Load datasets
train_dataset = datasets.ImageFolder(root='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid', transform=transform)

# DataLoader with batch size of 256 and 4 worker processes
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4)


class CustomCNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=5, padding=2)
        self.bn4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=5, padding=2)
        self.bn5 = nn.BatchNorm2d(512)

        self.res_conv=nn.Conv2d(32,256,kernel_size=3, padding=1)
        self.bnr = nn.BatchNorm2d(256)
        self.res_pool=nn.AdaptiveAvgPool2d((14,14))
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(512 * 7 * 7, 128)  # Adjust based on output size
        self.fc_out = nn.Linear(128, num_classes)
        self.activation = nn.ReLU()

    def forward(self, x):
        x1 = self.pool(self.activation(self.bn1(self.conv1(x))))  # (32, 112, 112)
        x2 = self.pool(self.activation(self.bn2(self.conv2(x1))))  # (64, 56, 56)
        x3 = self.pool(self.activation(self.bn3(self.conv3(x2))))  # (128, 28, 28)
        x4 = self.pool(self.activation(self.bn4(self.conv4(x3))))             # (256, 28, 28)

        x1_resized =self.res_pool(self.activation(self.res_conv(x1)))
        x_res = self.activation(x1_resized + x4)                                   # Residual connection
        x5 = self.pool(self.activation(self.bn5(self.conv5(x_res))))  # (128, 28, 28)
        x = x5.view(x5.size(0), -1)                                   # Flatten
        x = self.activation(self.fc1(x))
        x = self.fc_out(x)
        return x
    
# Initialize the model
num_classes = len(train_dataset.classes)
model = CustomCNNModel(num_classes=num_classes)

# Move model to the appropriate device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.AdamW(model.parameters(), lr=0.0043, weight_decay=1e-4)  # AdamW optimizer with weight decay

# Mixed precision training setup
scaler = torch.cuda.amp.GradScaler()

# Early stopping parameters
patience = 7
best_loss = float('inf')
epochs_without_improvement = 0

num_epochs = 40  # Number of epochs to train

# Record the start time
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    epoch_start_time = time.time()  # Record the start time for the epoch
    
    # Training phase
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # Mixed precision
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    
    train_loss = running_loss / len(train_dataset)
    
    # Validation phase
    model.eval()
    val_running_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():  # Mixed precision
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    val_loss = val_running_loss / len(val_dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    
    epoch_time = time.time() - epoch_start_time  # Calculate epoch duration
    print(f'Epoch {epoch + 1}/{num_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {accuracy:.4f}, Time: {epoch_time:.2f} seconds')
    
    # Check for early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        epochs_without_improvement = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping at epoch {epoch + 1}')
            break

# Record the total training time
total_time = time.time() - start_time
print(f'Total Training Time: {total_time:.2f} seconds')


Epoch 1/40, Training Loss: 3.7665, Validation Loss: 1.8197, Validation Accuracy: 0.4372, Time: 224.59 seconds


Epoch 2/40, Training Loss: 1.2711, Validation Loss: 0.8837, Validation Accuracy: 0.7260, Time: 223.43 seconds


Epoch 3/40, Training Loss: 0.7146, Validation Loss: 0.8950, Validation Accuracy: 0.7333, Time: 223.29 seconds


Epoch 4/40, Training Loss: 0.4376, Validation Loss: 0.6016, Validation Accuracy: 0.8132, Time: 222.07 seconds


Epoch 5/40, Training Loss: 0.3251, Validation Loss: 0.3066, Validation Accuracy: 0.9027, Time: 222.56 seconds


...

Epoch 20/40, Training Loss: 0.0238, Validation Loss: 0.1684, Validation Accuracy: 0.9585, Time: 223.50 seconds


Epoch 21/40, Training Loss: 0.0338, Validation Loss: 0.1239, Validation Accuracy: 0.9685, Time: 223.62 seconds


Epoch 22/40, Training Loss: 0.0237, Validation Loss: 0.2325, Validation Accuracy: 0.9492, Time: 222.41 seconds


Epoch 23/40, Training Loss: 0.0269, Validation Loss: 0.1330, Validation Accuracy: 0.9679, Time: 222.86 seconds


Epoch 24/40, Training Loss: 0.0296, Validation Loss: 0.0888, Validation Accuracy: 0.9773, Time: 224.35 seconds
Early stopping at epoch 24
Total Training Time: 5357.95 seconds
